In [1]:
import pandas as pd
import re
import glob
import os

def extract_financial_info(text):
    if not isinstance(text, str):
        return None, None, None

    # --- PATTERN 1: Keyword-based (эмитента ... ИНН ... ISIN) ---
    # Example: эмитента ООО ТК "Нафтатранс плюс" ИНН 5404345962 ... ISIN RU000A106Y21
    key_match = re.search(r'эмитента\s+(.*?)\s+ИНН\s+(\d{10,12}).*?ISIN\s+([A-Z0-9]{12})', text)
    if key_match:
        return key_match.group(1).strip(), key_match.group(2), key_match.group(3)

    # --- PATTERN 2: Structured Parentheses (Nested parentheses support) ---
    # Example: (Банк ВТБ (ПАО), 7702070139, RU000A10BJH2, ...)
    parentheses_blocks = re.findall(r'\(([^()]+(?:\([^()]*\)[^()]*)*)\)', text)
    if parentheses_blocks:
        # We look for a block containing a 10-12 digit INN or a 12-char ISIN
        for block in parentheses_blocks:
            parts = [p.strip() for p in block.split(',')]
            if len(parts) >= 3:
                # Validate if it looks like the Russian financial format
                if re.match(r'^\d{10,12}$', parts[1]):
                    name = parts[0].replace('""', '"').strip()
                    return name, parts[1], parts[2]

    # --- PATTERN 3: International/Sovereign (Hyphen based) ---
    # Example: - REPUBLIC OF BELARUS ... (облигация ... / ISIN XS...)
    # Adjusted to catch names even if the date is missing
    int_match = re.search(r'-\s*(.*?)\s*(?:\d{2}/\d{2}/\d{2,4})?\s*\(.*?ISIN\s+([A-Z0-9]{12})\)', text)
    if int_match:
        return int_match.group(1).strip(), None, int_match.group(2)

    # --- PATTERN 4: Disclosure Fallback (No ISIN/INN) ---
    # Example: эмитента ПАО АНК "Башнефть" (выпуск облигаций...)
    dscl_match = re.search(r'эмитента\s+(.*?)\s+\(выпуск облигаций', text)
    if dscl_match:
        return dscl_match.group(1).strip(), None, None

    return None, None, None

# --- FILE PROCESSING ---
file_pattern = os.path.join('CICs', '*_2025.csv')
files = glob.glob(file_pattern)

all_results = []

for file_path in files:
    try:
        df = pd.read_csv(file_path)
        if 'text' not in df.columns:
            continue
            
        # Apply extraction
        extracted_data = df['text'].apply(lambda x: pd.Series(extract_financial_info(x)))
        extracted_data.columns = ['company_name', 'inn', 'isin']
        
        # Merge with original data
        cols_to_keep = ['date', 'link', 'text'] # keeping text for validation
        result_df = pd.concat([df[df.columns.intersection(cols_to_keep)], extracted_data], axis=1)
        all_results.append(result_df)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

if all_results:
    final_df = pd.concat(all_results, ignore_index=True)
    # Output to CSV
    output_name = 'final_extracted_data_2025.csv'
    #final_df.to_csv(output_name, index=False, encoding='utf-8-sig')
    print(f"Extraction complete. {len(final_df)} rows saved to {output_name}")
else:
    print("No files were processed.")

Extraction complete. 14438 rows saved to final_extracted_data_2025.csv


In [2]:
final_df['date'] = pd.to_datetime(final_df['date'], format='%d.%m.%Y')
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14438 entries, 0 to 14437
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          14438 non-null  datetime64[ns]
 1   link          14438 non-null  object        
 2   text          14438 non-null  object        
 3   company_name  14413 non-null  object        
 4   inn           9153 non-null   object        
 5   isin          14385 non-null  object        
dtypes: datetime64[ns](1), object(5)
memory usage: 676.9+ KB


In [3]:
final_df[final_df['company_name'].isnull()]

,date,link,text,company_name,inn,isin
388,2025-04-28,https://nsddata.ru/ru/news/view/1250963,(INTR) (Выплата купонного дохода) О получении ...,None,None,None
1830,2025-04-02,https://nsddata.ru/ru/news/view/1242035,(INTR) (Выплата купонного дохода) О получении ...,None,None,None
1926,2025-05-30,https://nsddata.ru/ru/news/view/1264339,(INTR) (Выплата купонного дохода) О получении ...,None,None,None
3123,2025-05-12,https://nsddata.ru/ru/news/view/1256204,(INTR) (Выплата купонного дохода) О получении ...,None,None,None
3343,2025-05-05,https://nsddata.ru/ru/news/view/1253489,"(INTR) О корпоративном действии ""Выплата купон...",None,None,None
5381,2025-06-30,https://nsddata.ru/ru/news/view/1275985,(INTR) (Выплата купонного дохода) О получении ...,None,None,None
6062,2025-06-23,https://nsddata.ru/ru/news/view/1273653,(INTR) (Выплата купонного дохода) О получении ...,None,None,None
6063,2025-06-23,https://nsddata.ru/ru/news/view/1273652,(INTR) (Выплата купонного дохода) О получении ...,None,None,None
6064,2025-06-23,https://nsddata.ru/ru/news/view/1273651,(INTR) (Выплата купонного дохода) О получении ...,None,None,None
6065,2025-06-23,https://nsddata.ru/ru/news/view/1273647,(INTR) (Выплата купонного дохода) О получении ...,None,None,None


### Finding issuers active during February, March, April, May, June, July, August and September

In [4]:
final_df['month'] = pd.to_datetime(final_df['date']).dt.month
final_df['month'].unique()

array([4, 5, 3, 6, 2, 8, 9, 7])

In [5]:
# The required months
required_months = {2, 3, 4, 5, 6, 7, 8, 9}

# Get unique months per company from final_df
company_month_counts = final_df.groupby('company_name')['month'].unique()

# Filter companies whose unique months contain all required months
companies_with_all_months = []
for company, months in company_month_counts.items():
    if required_months.issubset(set(months)):
        companies_with_all_months.append(company)

print(f'Number of companies with all 8 months: {len(companies_with_all_months)}')
print("Companies with all 8 months:", companies_with_all_months)


Number of companies with all 8 months: 125
Companies with all 8 months: ['BBVA Global Markets B.V.', 'Citigroup Global Markets Funding Luxembourg S.C.A.', 'EFG International Finance', 'GOLDMAN SACHS INT', 'GOLDMAN SACHS INTERNATIONAL', 'Goldman Sachs Finance Corp International Ltd', 'Goldman Sachs International', 'HSBC Bank PLC', 'HSBC Bank Plc', 'J.P. Morgan Chase Bank, N.A.', 'JPMORGAN CHASE BANK', 'JPMORGAN CHASE BANK, N.A.', 'Leonteq Securities AG, Guernsey Branch', 'Marex Financial', 'Mikro Fund F.T.', 'Mikro Fund F.T. 6.0', 'Mikro Fund F.T. 6.00', 'Mikro Fund F.T. 7.00', 'Morgan Stanley & Co International PLC', 'Morgan Stanley B.V.', 'UBS AG, London Branch', 'АО "АЛЬФА-БАНК"', 'АО "БИЗНЕС АЛЬЯНС"', 'АО "ГТЛК"', 'АО "ДОМ.РФ"', 'АО "Кириллица"', 'АО "Коммерческая недвижимость ФПК "Гарант-Инвест"', 'АО "МОНОПОЛИЯ"', 'АО "Полипласт"', 'АО "РОЛЬФ"', 'АО "Росагролизинг"', 'АО "Россельхозбанк"', 'АО "Сбербанк КИБ"', 'АО "ТАЛК лизинг"', 'АО "ХК "МЕТАЛЛОИНВЕСТ"', 'АО "Хвоя"', 'АО ВТБ Лизи

In [28]:
foreign_names = ['BBVA Global Markets B.V.',
 'Citigroup Global Markets Funding Luxembourg S.C.A.',
 'EFG International Finance',
 'GOLDMAN SACHS INT',
 'GOLDMAN SACHS INTERNATIONAL',
 'Goldman Sachs Finance Corp International Ltd',
 'Goldman Sachs International',
 'HSBC Bank PLC',
 'HSBC Bank Plc',
 'J.P. Morgan Chase Bank, N.A.',
 'JPMORGAN CHASE BANK',
 'JPMORGAN CHASE BANK, N.A.',
 'Leonteq Securities AG, Guernsey Branch',
 'Marex Financial',
 'Mikro Fund F.T.',
 'Mikro Fund F.T. 6.0',
 'Mikro Fund F.T. 6.00',
 'Mikro Fund F.T. 7.00',
 'Morgan Stanley & Co International PLC',
 'Morgan Stanley B.V.',
 'UBS AG, London Branch']

In [29]:
data = pd.DataFrame()
for i in companies_with_all_months:
    name = i
    # Skip if the name is in foreign_names list
    if name not in foreign_names:
        data = pd.concat([data, final_df[final_df['company_name'] == i]])

In [33]:
data.to_csv('coupons_to_scrape.csv')